In [7]:
import os

def setting_up_proxy(proxy=None, proxy_type='http', verbose=True):
    supported_proxy_types = ['http', 'https', 'socks4', 'socks5', 'all']
    assert proxy_type in supported_proxy_types, f"proxy type {repr(proxy_type)} not supported, only support {supported_proxy_types}"
    if proxy is None:
        proxy = os.environ.get(f'{proxy_type}_proxy')
    if proxy is None:
        return
    if verbose:
        print(f'setting up proxy {repr(proxy)} for {repr(proxy_type)}')
    os.environ[f'{proxy_type}_proxy'] = proxy


default_proxy_config = {
    'http': 'http://10.176.52.116:7890',
    'https': 'http://10.176.52.116:7890',
    'all': 'socks5://10.176.52.116:7890',
}


def setting_up_proxy_from_config(proxy_config=default_proxy_config, verbose=True):
    for proxy_type, proxy_url in proxy_config.items():
        setting_up_proxy(proxy=proxy_url, proxy_type=proxy_type, verbose=verbose)
    print()


# setting_up_proxy_from_config()

In [8]:
import requests
resp = requests.get('https://audiostock.jp/audio/1536493', proxies=default_proxy_config)
print(resp)
# print(resp.content)
print(resp.headers['content-type'])
file_type = 'html'
with open(f'test.{file_type}', 'wb') as f:
    f.write(resp.content)

<Response [200]>
text/html; charset=utf-8


In [34]:
import urllib.parse as urlparse
import requests
import json


with open('examples/1536493.html', 'r') as f:
    html = f.read()

from RFC.utils.parse import (
    get_html_soup,
    parse,
)

soup = get_html_soup(html)

parse_config = {
    'audio_tags': {
        ('attr', 'div', 'class', 'list-tag', None): {
            ('type', 'dd', None): {
                # ('attr', 'a', 'class', 'btn tag-normal', None): {
                #     # ('result', 'text', None): {},
                # },
            }
        },
    },
    'audio_tag_types': {
        ('attr', 'div', 'class', 'list-tag', None): {
            ('type', 'dt', None): {
                ('result', 'text', None): {},
            }
        },
    },
    'audio_link': {
        ('attr', 'span', 'class', 'player-audio-left-btn play-button', 0): {},
    },
    'audio_tags_in_each_group': {
        ('attr', 'a', 'class', 'btn tag-normal', None): {
            # ('result', 'text', None): {},
            # ('result', 'href', None): {},
        },
    },
}

audio_link = parse(soup, parse_config['audio_link'], return_str=False, debug=False)[0]['data-audio_url']
tag_types = parse(soup, parse_config['audio_tag_types'], return_str=False, debug=False)
tag_groups = parse(soup, parse_config['audio_tags'], return_str=False, debug=False)
# print(tag_types, tag_groups)
assert len(tag_types) == len(tag_groups)
parsed_tags = []
for tag_group in tag_groups:
    tags = parse(tag_group, parse_config['audio_tags_in_each_group'], return_str=False, debug=False)
    tags_link = [tag['href'] for tag in tags]
    tags_text = [tag.text for tag in tags]
    assert len(tags_link) == len(tags_text)
    tags = list(zip(tags_text, tags_link))
    parsed_tags.append(tags)
assert len(parsed_tags) == len(tag_types)
tag_result = {tag_type: tags for tag_type, tags in zip(tag_types, parsed_tags)}
result = {
    'audio_link': audio_link,
    'audio_tags': tag_result,
}
print(json.dumps(result, indent=4, ensure_ascii=False))

{
    "audio_link": "https://audiostock.jp/audio/1536493/play?no-cache=14cb66e18a4be80efb5dc48a24b7d09f",
    "audio_tags": {
        "用途": [
            [
                "映像・動画",
                "https://audiostock.jp/bgm/763"
            ],
            [
                "アニメ",
                "https://audiostock.jp/bgm/765"
            ],
            [
                "CM（コマーシャル）",
                "https://audiostock.jp/bgm/766"
            ],
            [
                "料理（クッキング）",
                "https://audiostock.jp/bgm/785"
            ]
        ],
        "楽器": [
            [
                "アコースティックギター（アコギ）",
                "https://audiostock.jp/bgm/817"
            ],
            [
                "バイオリン",
                "https://audiostock.jp/bgm/821"
            ],
            [
                "民族楽器",
                "https://audiostock.jp/bgm/835"
            ],
            [
                "クラリネット",
                "https://audiostock.jp/bgm/1681"
            

In [4]:
import requests
server = '222.184.170.99:32681'
proxy_url = f'http://D6FL1CJ8:9FC1ADB88090@{server}'
proxies = {
    "http": proxy_url,
    "https": proxy_url,
}
resp = requests.get('https://audiostock.jp/audio/1536493/play?no-cache=8609d2970d99e7ae443c1c8f10362a95', proxies=proxies)
print(resp)
# print(resp.content)
print(resp.headers['content-type'])
file_type = resp.headers['content-type'].split('/')[-1]
with open(f'test.{file_type}', 'wb') as f:
    f.write(resp.content)